# Building Your First Network Exercises

Write your answer in the empty code cell under each question. Check yourself against `solutions.ipynb` when done.

**Part 1** = warm-up · **Part 2** = practice · **Part 3** = challenge.

## Part 1 — Warm-up

**1. Meet the two moons.** Run `X, y = make_moons(n_samples=200, noise=0.15, random_state=42)` (import from `sklearn.datasets`). Print the shapes of `X` and `y`, the unique classes, how many samples each class has (`np.unique(y, return_counts=True)`) and the first three rows of `X` rounded to 3 decimals.

**2. Split and standardise (no leakage).** Continuing from the moons data: seed `rng = np.random.default_rng(42)`, shuffle with `idx = rng.permutation(len(X))` and reorder `X, y`. Take the first 150 rows as train and the remaining 50 as test, reshaping each `y` to `(-1, 1)` as float. Compute `mu` and `sd` from the TRAINING rows ONLY and standardise both splits with them. Print both shapes and the train column means after scaling.

**3. Seeded He initialisation.** With `rng = np.random.default_rng(7)` and `HIDDEN = 16`, create `W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, HIDDEN))`, `b1 = np.zeros((1, HIDDEN))`, `W2 = rng.normal(0, np.sqrt(2 / HIDDEN), size=(HIDDEN, 1))`, `b2 = np.zeros((1, 1))`. Print every shape and std, then count the total trainable parameters with `sum(p.size for p in [W1, b1, W2, b2])`. Why must the weights be random rather than all zeros?

## Part 2 — Practice

**4. Wrap the forward pass in functions.** Using the parameters from the previous question, define `forward(X)` returning `Z1 = X @ W1 + b1`, `A1 = np.maximum(Z1, 0)` and `Z2 = A1 @ W2 + b2`, plus a `sigmoid(z)` helper. Run them on `demo = np.array([[0.0, 1.0], [1.5, -0.5]])`; print how many hidden units survived ReLU (of all units), the logits and the probabilities, each rounded to 4 decimals.

**5. Stable BCE on raw logits.** Implement `bce_with_logits(z, y) = np.mean(np.maximum(z, 0) - y * z + np.log1p(np.exp(-np.abs(z))))`. Test it with `y_true = np.array([[1.0], [0.0]])` against three logit sets: `good = np.array([[3.0], [-3.0]])`, `zero = np.array([[0.0], [0.0]])` and `bad = np.array([[-3.0], [3.0]])`. Print each loss alongside the probabilities `sigmoid(z)`. What does a completely UNDECIDED model pay?

**6. Seven lines of backprop.** Rebuild a tiny net: `rng = np.random.default_rng(7)`, `W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, 8))`, `b1 = np.zeros((1, 8))`, `W2 = rng.normal(0, np.sqrt(2 / 8), size=(8, 1))`, `b2 = np.zeros((1, 1))`. For `Xs = np.array([[0.5, -1.0], [1.5, 0.5]])` and `ys = np.array([[1.0], [0.0]])`, run the forward pass KEEPING `Z1, A1, Z2`, then compute all seven gradients: `dZ2 = (sigmoid(Z2) - ys) / N`, `dW2 = A1.T @ dZ2`, `db2 = dZ2.sum(0, keepdims=True)`, `dA1 = dZ2 @ W2.T`, `dZ1 = dA1 * (Z1 > 0)`, `dW1 = Xs.T @ dZ1`, `db1 = dZ1.sum(0, keepdims=True)`. Print each gradient's shape next to its parameter's shape.

**7. Check the BCE gradient identity.** The lesson claims the gradient of BCE-with-logits wrt a logit is `(sigmoid(z) - y) / N`. Verify it: with `y_true = np.array([[1.0], [0.0]])` and `z = np.array([[0.5], [-1.0]])`, use your `bce_with_logits` from Question 5, perturb EACH logit by `eps = 1e-6` up and down, compute the central-difference derivative, and compare against `(sigmoid(z) - y_true) / 2`. Print both vectors and their maximum absolute difference.

## Part 3 — Challenge

**8. Train the moons network.** Assemble the full pipeline with the SAME seeds throughout: moons data split and standardised as in Question 2, He init from Question 3 (`default_rng(7)`, `HIDDEN = 16`), the forward pass, stable BCE loss, the seven-gradient backward pass and updates `W -= lr * dW` with `lr = 0.5` for 300 full-batch epochs. Print the loss every 100 epochs. Afterwards evaluate ONCE on the untouched test set with `accuracy = ((Z2_test.ravel() > 0) == y_test.ravel()).mean()` and compare it to the ~85% ceiling of any straight line.

**9. Learning-rate experiment.** Wrap the Question-8 pipeline in a function `train(lr, epochs=300)` that builds a FRESH He-initialised network (same `default_rng(7)` seed each call so every run starts identically), trains it, and returns the final training loss and test accuracy. Run it for `lr` in `[0.05, 0.5, 20.0]` and print one line per rate. In comments: which rate crawls, which converges smoothly, and which overshoots so badly that accuracy drops BELOW chance?